# SODIndoorLoc: MLP centralizado, FedAvg y modelos por clúster

Usa las huellas RAW de HCXY y evalúa únicamente coordenadas porque el subconjunto seleccionado contiene una sola planta.


## 1. Objetivo y protocolo experimental

Se comparan cuatro modalidades de MLP: entrenamiento centralizado
global, entrenamiento federado global mediante FedAvg,
entrenamiento centralizado por clúster predicho y entrenamiento federado
por clúster predicho. Los hiperparámetros y el número de clústeres se
seleccionan únicamente con validación. Test se evalúa después de fijar
cada ganador.


## 2. Importaciones y configuración del entorno


In [1]:
from pathlib import Path
from dataclasses import asdict, dataclass
from itertools import product
from typing import Dict, List, Mapping, Optional, Sequence, Tuple
import copy
import json
import math
import random
import re
import warnings

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.cluster import KMeans
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.neighbors import (
    KNeighborsClassifier,
    KNeighborsRegressor,
    NearestNeighbors,
)
from sklearn.preprocessing import LabelEncoder, StandardScaler

try:
    from IPython.display import display
except ImportError:
    display = print


# Si Jupyter se inicia fuera de la carpeta del paquete, escribe aquí su ruta.
# Ejemplo: PROJECT_ROOT = Path(r"C:/TFM/notebooks_autocontenidos_sin_core")
PROJECT_ROOT = None

cwd = Path.cwd().resolve()
root_candidates = [cwd, cwd.parent, cwd.parent.parent]
if PROJECT_ROOT is not None:
    ROOT = Path(PROJECT_ROOT).expanduser().resolve()
else:
    ROOT = next(
        (
            candidate
            for candidate in root_candidates
            if sum((candidate / name).is_dir() for name in ["TUT", "TUJI1", "UJIIndoor", "SOD"])
            >= 2
        ),
        cwd,
    )

SEED = 42
TARGET_COLUMNS = ["TARGET_X_M", "TARGET_Y_M"]

print("Raíz utilizada:", ROOT)


Raíz utilizada: /home/coder/Indoor/Notebooks


### 2.1. Carga y preprocesado

Estas funciones están dentro del notebook. Detectan las columnas
RSSI, cargan las particiones creadas y aplican
una transformación ajustada exclusivamente con `train`.


In [2]:
def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch

        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def natural_key(text: str) -> List[object]:
    return [int(piece) if piece.isdigit() else piece for piece in re.split(r"(\d+)", text)]


def detect_rssi_columns(df: pd.DataFrame) -> List[str]:
    cols = [c for c in df.columns if re.fullmatch(r"(?:WAP|MAC)\d+", str(c).upper())]
    cols = sorted(cols, key=natural_key)
    if not cols:
        raise ValueError("No se detectaron columnas RSSI WAPnnn o MACnnn.")
    return cols


class RSSIPreprocessor:
    """Imputa ausencias, estandariza RSSI con train y anade mascara de deteccion."""

    def __init__(self, missing_value: float = 100.0, fill_value: float = -110.0, use_mask: bool = True):
        self.missing_value = float(missing_value)
        self.fill_value = float(fill_value)
        self.use_mask = bool(use_mask)
        self.scaler = StandardScaler()
        self.columns: List[str] = []

    def _clean(self, df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
        raw = df[self.columns].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
        observed = np.isfinite(raw) & (raw != self.missing_value)
        clean = np.where(observed, raw, self.fill_value).astype(np.float32)
        return clean, observed.astype(np.float32)

    def fit(self, df: pd.DataFrame, columns: Sequence[str]) -> "RSSIPreprocessor":
        self.columns = list(columns)
        clean, _ = self._clean(df)
        self.scaler.fit(clean)
        return self

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        clean, mask = self._clean(df)
        scaled = self.scaler.transform(clean).astype(np.float32)
        if self.use_mask:
            return np.concatenate([scaled, mask], axis=1).astype(np.float32)
        return scaled

    def fit_transform(self, df: pd.DataFrame, columns: Sequence[str]) -> np.ndarray:
        return self.fit(df, columns).transform(df)


#### Lectura de particiones y rutas


In [3]:
def read_base_splits(output_dir: Path, prefix: str) -> Dict[str, pd.DataFrame]:
    output_dir = Path(output_dir)
    return {
        split: pd.read_csv(output_dir / f"{prefix}_{split}.csv")
        for split in ["train", "val", "test"]
    }


def load_prepared_bundle(prepared_dir: Path, prefix: str) -> Tuple[Dict[str, pd.DataFrame], pd.DataFrame]:
    prepared_dir = Path(prepared_dir)
    splits = read_base_splits(prepared_dir, prefix)
    routes = pd.read_csv(prepared_dir / f"{prefix}_routes.csv")
    expected = {"ROW_ID", "SPLIT", "N_CLUSTERS", "CLUSTER", "CLUSTER_ORACLE"}
    if not expected.issubset(routes.columns):
        raise ValueError(f"El fichero de rutas no contiene {sorted(expected)}")
    return splits, routes


def route_frame(frame: pd.DataFrame, routes: pd.DataFrame, split: str, n_clusters: int) -> pd.DataFrame:
    selected = routes[
        (routes["SPLIT"].astype(str) == split)
        & (pd.to_numeric(routes["N_CLUSTERS"]) == int(n_clusters))
    ].copy()
    out = frame.merge(selected, on="ROW_ID", how="left", validate="one_to_one")
    if out["CLUSTER"].isna().any():
        raise ValueError(f"Faltan rutas para {split}, K={n_clusters}.")
    out["CLUSTER"] = out["CLUSTER"].astype(int)
    out["CLUSTER_ORACLE"] = out["CLUSTER_ORACLE"].astype(int)
    return out


### 2.2. Métricas y enrutamiento por clúster

El error de coordenadas es la distancia radial 2D en metros. Para
los modelos por zona, `CLUSTER` es la salida del router RSSI;
`CLUSTER_ORACLE` solo se utiliza como diagnóstico.


In [4]:
def coordinate_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    truth = np.asarray(y_true, dtype=float)
    pred = np.asarray(y_pred, dtype=float)
    if truth.shape != pred.shape or truth.ndim != 2 or truth.shape[1] != 2:
        raise ValueError(f"Se esperaban matrices (n,2); recibidas {truth.shape} y {pred.shape}.")
    distances = np.linalg.norm(pred - truth, axis=1)
    return {
        "rmse_2d_m": float(np.sqrt(np.mean(distances ** 2))),
        "mean_error_m": float(np.mean(distances)),
        "median_error_m": float(np.median(distances)),
        "p75_error_m": float(np.quantile(distances, 0.75)),
        "p95_error_m": float(np.quantile(distances, 0.95)),
        "n": int(len(distances)),
    }


def _metric_row(
    model: str,
    modality: str,
    split: str,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    n_clusters: Optional[int] = None,
    extra: Optional[Mapping[str, object]] = None,
) -> Dict[str, object]:
    row: Dict[str, object] = {
        "model": model,
        "modality": modality,
        "split": split,
        "n_clusters": n_clusters,
        **coordinate_metrics(y_true, y_pred),
    }
    if extra:
        row.update(dict(extra))
    return row


def _predict_by_zone(
    models: Mapping[int, object],
    fallback: object,
    x: np.ndarray,
    zone_labels: np.ndarray,
) -> Tuple[np.ndarray, float]:
    pred = np.empty((len(x), 2), dtype=float)
    covered = np.zeros(len(x), dtype=bool)
    zones = np.asarray(zone_labels, dtype=int)
    for zone in np.unique(zones):
        mask = zones == zone
        model = models.get(int(zone), fallback)
        pred[mask] = model.predict(x[mask])
        covered[mask] = int(zone) in models
    return pred, float(np.mean(covered))


def compact_results(result: pd.DataFrame, split: str = "test") -> pd.DataFrame:
    columns = [
        "model",
        "modality",
        "n_clusters",
        "rmse_2d_m",
        "mean_error_m",
        "median_error_m",
        "p75_error_m",
        "p95_error_m",
        "coverage",
        "gate_accuracy_diagnostic",
    ]
    available = [c for c in columns if c in result.columns]
    return result[result["split"] == split][available].sort_values("rmse_2d_m").reset_index(drop=True)


## 3. Ficheros y configuración del dataset


In [5]:
DATASET = 'SOD'
MODEL_FAMILY = 'MLP'
PREPARED_DIR = ROOT / "prepared" / 'SOD'
PREFIX = 'sod_raw'
RESULTS_DIR = ROOT / "results" / 'SOD'
CLUSTER_VALUES = [2, 3, 4, 5, 6, 7, 8, 9, 10]
FLOOR_TASK = False
REQUIRE_KNOWN_TEST_CLIENTS = False

COORD_RESULTS_PATH = RESULTS_DIR / 'sod_raw_mlp_corrected.csv'
COORD_HPARAM_TUNING_PATH = COORD_RESULTS_PATH.with_name(
    COORD_RESULTS_PATH.stem + "_hyperparameter_tuning.csv"
)
COORD_CLUSTER_TUNING_PATH = COORD_RESULTS_PATH.with_name(
    COORD_RESULTS_PATH.stem + "_cluster_tuning.csv"
)
COORD_SELECTED_PATH = COORD_RESULTS_PATH.with_name(
    COORD_RESULTS_PATH.stem + "_selected_hyperparameters.json"
)

if FLOOR_TASK:
    FLOOR_RESULTS_PATH = RESULTS_DIR / ''
    FLOOR_HPARAM_TUNING_PATH = FLOOR_RESULTS_PATH.with_name(
        FLOOR_RESULTS_PATH.stem + "_hyperparameter_tuning.csv"
    )
    FLOOR_CLUSTER_TUNING_PATH = FLOOR_RESULTS_PATH.with_name(
        FLOOR_RESULTS_PATH.stem + "_cluster_tuning.csv"
    )
    FLOOR_SELECTED_PATH = FLOOR_RESULTS_PATH.with_name(
        FLOOR_RESULTS_PATH.stem + "_selected_hyperparameters.json"
    )


In [6]:
required_inputs = [
    PREPARED_DIR / f"{PREFIX}_train.csv",
    PREPARED_DIR / f"{PREFIX}_val.csv",
    PREPARED_DIR / f"{PREFIX}_test.csv",
    PREPARED_DIR / f"{PREFIX}_routes.csv",
]
missing_inputs = [path for path in required_inputs if not path.exists()]

if missing_inputs:
    formatted = "\n- ".join(str(path) for path in missing_inputs)
    raise FileNotFoundError(
        "Faltan las particiones preparadas:\n- " + formatted
        + "\nEjecuta primero el notebook 00 del dataset incluido en este paquete."
    )

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Raíz del proyecto:", ROOT)
print("Datos preparados:", PREPARED_DIR)
print("Resultados:", RESULTS_DIR)


Raíz del proyecto: /home/coder/Indoor/Notebooks
Datos preparados: /home/coder/Indoor/Notebooks/prepared/SOD
Resultados: /home/coder/Indoor/Notebooks/results/SOD


## 4. Carga de datos

Las particiones y las rutas RSSI→clúster del
dataset. Validación y test no reutilizan clústeres calculados a partir de
sus coordenadas reales.


In [7]:
splits, routes = load_prepared_bundle(PREPARED_DIR, PREFIX)

position_columns = ["TARGET_X_M", "TARGET_Y_M"]
if "FLOOR_LABEL" in splits["train"].columns:
    position_columns.append("FLOOR_LABEL")

partition_summary = []
for split_name, frame in splits.items():
    partition_summary.append(
        {
            "split": split_name,
            "rows": len(frame),
            "positions": frame[position_columns].drop_duplicates().shape[0],
            "clients": frame["CLIENT_ID"].astype(str).nunique(),
            "floors": (
                frame["FLOOR_LABEL"].nunique()
                if "FLOOR_LABEL" in frame.columns
                else np.nan
            ),
        }
    )

display(pd.DataFrame(partition_summary).set_index("split"))
print("Valores de K disponibles:", sorted(routes["N_CLUSTERS"].astype(int).unique()))


,rows,positions,clients,floors
split,,,,
train,9660,322,6,NaN
val,1710,57,6,NaN
test,860,86,6,NaN


Valores de K disponibles: [np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]


## 5. Auditoría de las particiones


In [8]:
def _row_ids(frame):
    return set(frame["ROW_ID"].astype(str))


def _positions(frame):
    return set(map(tuple, frame[position_columns].astype(str).to_numpy()))


audit_rows = []
for left, right in [("train", "val"), ("train", "test"), ("val", "test")]:
    audit_rows.append(
        {
            "partitions": f"{left}-{right}",
            "shared_row_ids": len(_row_ids(splits[left]) & _row_ids(splits[right])),
            "shared_positions": len(_positions(splits[left]) & _positions(splits[right])),
        }
    )

display(pd.DataFrame(audit_rows).set_index("partitions"))

train_clients = set(splits["train"]["CLIENT_ID"].astype(str))
val_clients = set(splits["val"]["CLIENT_ID"].astype(str))
test_clients = set(splits["test"]["CLIENT_ID"].astype(str))
unseen_vs_train = sorted(test_clients - train_clients)
unseen_vs_train_val = sorted(test_clients - (train_clients | val_clients))

print("Clientes de test ausentes en train:", unseen_vs_train)
print("Clientes de test ausentes en train/validación:", unseen_vs_train_val)

assert len(_row_ids(splits["train"]) & _row_ids(splits["val"])) == 0
assert len(_positions(splits["train"]) & _positions(splits["val"])) == 0
if REQUIRE_KNOWN_TEST_CLIENTS:
    assert unseen_vs_train_val == [], (
        "UJIIndoorLoc contiene dispositivos de test no presentes en train/validación: "
        f"{unseen_vs_train_val}. Vuelve a ejecutar el notebook 00 corregido."
    )

print("Auditoría superada.")


,shared_row_ids,shared_positions
partitions,,
train-val,0,0
train-test,0,0
val-test,0,0


Clientes de test ausentes en train: []
Clientes de test ausentes en train/validación: []
Auditoría superada.


## 6. Variables disponibles


In [9]:
rssi_columns = detect_rssi_columns(splits["train"])

variable_summary = pd.DataFrame(
    {
        "group": ["RSSI", "target", "client", "routing"],
        "columns": [
            len(rssi_columns),
            len(
                [
                    column
                    for column in ["TARGET_X_M", "TARGET_Y_M", "FLOOR_LABEL"]
                    if column in splits["train"]
                ]
            ),
            1,
            len(
                [
                    column
                    for column in ["ROW_ID", "SPLIT"]
                    if column in splits["train"]
                ]
            ),
        ],
    }
)
display(variable_summary.set_index("group"))
print("Primeras columnas RSSI:", rssi_columns[:10])
print(
    "Rango RSSI en train:",
    float(splits["train"][rssi_columns].min().min()),
    "a",
    float(splits["train"][rssi_columns].max().max()),
)


,columns
group,
RSSI,347
target,2
client,1
routing,1


Primeras columnas RSSI: ['MAC1', 'MAC2', 'MAC3', 'MAC4', 'MAC5', 'MAC6', 'MAC7', 'MAC8', 'MAC9', 'MAC10']
Rango RSSI en train: -98.0 a 100.0


## 7. Variables objetivo y métricas

La regresión usa `TARGET_X_M` y `TARGET_Y_M`. La selección se realiza con
RMSE radial 2D en metros y también se guardan error medio, mediana, P75 y
P95. Cuando existe `FLOOR_LABEL`, la clasificación de planta constituye
una tarea independiente y se selecciona mediante accuracy.


## 8. Preprocesado RSSI


In [10]:
# Esta celda documenta las dimensiones. Las funciones de entrenamiento vuelven
# a ajustar internamente un preprocesador idéntico usando exclusivamente train.
preprocessor_preview = RSSIPreprocessor(use_mask=True).fit(splits["train"], rssi_columns)
feature_shapes = {
    split_name: preprocessor_preview.transform(frame).shape
    for split_name, frame in splits.items()
}
display(
    pd.DataFrame(
        [
            {"split": name, "samples": shape[0], "features_after_mask": shape[1]}
            for name, shape in feature_shapes.items()
        ]
    ).set_index("split")
)
print("La imputación y el escalador RSSI se ajustan únicamente con train.")


,samples,features_after_mask
split,,
train,9660,694
val,1710,694
test,860,694


La imputación y el escalador RSSI se ajustan únicamente con train.


## 9. Modelo MLP y estrategia federada

Para coordenadas se usa una red totalmente conectada con dos salidas,
correspondientes a `TARGET_X_M` y `TARGET_Y_M`. Las coordenadas objetivo
se estandarizan con train y se devuelven a metros antes de calcular las
métricas. Para planta, la última capa se sustituye por una salida
multiclase y se optimiza con entropía cruzada.

La modalidad federada aplica FedAvg real: cada cliente parte del modelo
global, realiza entrenamiento local y el servidor promedia los parámetros.
Se comparan ponderación uniforme y ponderación por número de muestras.
Validación controla la selección y la parada; test no interviene.


### 9.1. Implementación autocontenida de MLP

Aquí se definen el modelo central, su estrategia distribuida o
federada y el procedimiento completo de selección con validación.
No se importa código propio desde ningún fichero `.py`.


#### Arquitectura y configuración MLP


In [11]:
@dataclass
class MLPConfig:
    hidden_sizes: Tuple[int, ...] = (256, 128)
    dropout: float = 0.10
    learning_rate: float = 1e-3
    weight_decay: float = 1e-5
    batch_size: int = 128
    central_epochs: int = 100
    patience: int = 15
    federated_rounds: int = 40
    local_epochs: int = 2
    client_weighting: str = "uniform"
    force_cpu: bool = True
    seed: int = SEED


def _torch_components(input_dim: int, config: MLPConfig):
    import torch
    import torch.nn as nn

    layers: List[nn.Module] = []
    last = input_dim
    for width in config.hidden_sizes:
        layers.extend([nn.Linear(last, width), nn.ReLU(), nn.Dropout(config.dropout)])
        last = width
    layers.append(nn.Linear(last, 2))
    return nn.Sequential(*layers)


def _torch_predict(model, x: np.ndarray, device, batch_size: int = 2048) -> np.ndarray:
    import torch

    model.eval()
    parts = []
    with torch.no_grad():
        for start in range(0, len(x), batch_size):
            tensor = torch.as_tensor(x[start : start + batch_size], dtype=torch.float32, device=device)
            parts.append(model(tensor).detach().cpu().numpy())
    return np.concatenate(parts, axis=0) if parts else np.empty((0, 2), dtype=float)


class TorchCoordRegressor:
    def __init__(self, model, target_scaler: StandardScaler, device):
        self.model = model
        self.target_scaler = target_scaler
        self.device = device

    def predict(self, x: np.ndarray) -> np.ndarray:
        scaled = _torch_predict(self.model, x, self.device)
        return self.target_scaler.inverse_transform(scaled)


#### Entrenamiento MLP centralizado


In [12]:
def _train_torch_local(model, x, y, config: MLPConfig, device, epochs: int):
    import torch
    from torch.utils.data import DataLoader, TensorDataset

    model.train()
    dataset = TensorDataset(
        torch.as_tensor(x, dtype=torch.float32), torch.as_tensor(y, dtype=torch.float32)
    )
    generator = torch.Generator().manual_seed(config.seed)
    loader = DataLoader(
        dataset,
        batch_size=min(config.batch_size, max(1, len(dataset))),
        shuffle=True,
        generator=generator,
    )
    optimizer = torch.optim.Adam(
        model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay
    )
    loss_fn = torch.nn.MSELoss()
    for _ in range(int(epochs)):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
    return model


def _fit_mlp_central(
    x_train: np.ndarray,
    y_train_scaled: np.ndarray,
    x_val: np.ndarray,
    y_val_scaled: np.ndarray,
    target_scaler: StandardScaler,
    config: MLPConfig,
):
    import torch
    from torch.utils.data import DataLoader, TensorDataset

    device = torch.device("cpu" if config.force_cpu or not torch.cuda.is_available() else "cuda")
    model = _torch_components(x_train.shape[1], config).to(device)
    dataset = TensorDataset(
        torch.as_tensor(x_train, dtype=torch.float32),
        torch.as_tensor(y_train_scaled, dtype=torch.float32),
    )
    generator = torch.Generator().manual_seed(config.seed)
    loader = DataLoader(
        dataset,
        batch_size=min(config.batch_size, max(1, len(dataset))),
        shuffle=True,
        generator=generator,
    )
    optimizer = torch.optim.Adam(
        model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay
    )
    loss_fn = torch.nn.MSELoss()
    best_state = copy.deepcopy(model.state_dict())
    best_val = float("inf")
    stale = 0
    for _ in range(config.central_epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
        val_pred = _torch_predict(model, x_val, device)
        val_loss = float(np.mean((val_pred - y_val_scaled) ** 2))
        if val_loss < best_val - 1e-8:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1
            if stale >= config.patience:
                break
    model.load_state_dict(best_state)
    return TorchCoordRegressor(model, target_scaler, device)


#### Entrenamiento federado con FedAvg


In [13]:
def _average_state_dicts(states, weights):
    total = float(sum(weights))
    output = copy.deepcopy(states[0])
    for key in output:
        output[key] = sum(state[key] * (float(weight) / total) for state, weight in zip(states, weights))
    return output


def _fit_mlp_fedavg(
    x_train: np.ndarray,
    y_train_scaled: np.ndarray,
    clients: np.ndarray,
    x_val: np.ndarray,
    y_val_scaled: np.ndarray,
    target_scaler: StandardScaler,
    config: MLPConfig,
):
    import torch

    device = torch.device("cpu" if config.force_cpu or not torch.cuda.is_available() else "cuda")
    global_model = _torch_components(x_train.shape[1], config).to(device)
    best_state = copy.deepcopy(global_model.state_dict())
    best_val = float("inf")
    stale = 0
    client_ids = np.asarray(clients).astype(str)
    unique_clients = sorted(np.unique(client_ids))
    for _round in range(config.federated_rounds):
        states = []
        weights = []
        for client in unique_clients:
            mask = client_ids == client
            if not np.any(mask):
                continue
            local_model = _torch_components(x_train.shape[1], config).to(device)
            local_model.load_state_dict(copy.deepcopy(global_model.state_dict()))
            _train_torch_local(
                local_model, x_train[mask], y_train_scaled[mask], config, device, config.local_epochs
            )
            states.append({k: v.detach().clone() for k, v in local_model.state_dict().items()})
            weights.append(1.0 if config.client_weighting == "uniform" else float(np.sum(mask)))
        global_model.load_state_dict(_average_state_dicts(states, weights))
        val_pred = _torch_predict(global_model, x_val, device)
        val_loss = float(np.mean((val_pred - y_val_scaled) ** 2))
        if val_loss < best_val - 1e-8:
            best_val = val_loss
            best_state = copy.deepcopy(global_model.state_dict())
            stale = 0
        else:
            stale += 1
            if stale >= config.patience:
                break
    global_model.load_state_dict(best_state)
    return TorchCoordRegressor(global_model, target_scaler, device)


#### Selección y evaluación MLP


In [14]:
def run_mlp_experiment(
    prepared_dir: Path,
    prefix: str,
    config: Optional[MLPConfig] = None,
    central_configs: Optional[Sequence[MLPConfig]] = None,
    federated_configs: Optional[Sequence[MLPConfig]] = None,
    cluster_values: Optional[Sequence[int]] = None,
    output_csv: Optional[Path] = None,
) -> pd.DataFrame:
    fallback_config = config or MLPConfig()
    central_candidates = (
        list(central_configs) if central_configs is not None else [fallback_config]
    )
    federated_candidates = (
        list(federated_configs) if federated_configs is not None else [fallback_config]
    )
    if not central_candidates or not federated_candidates:
        raise ValueError("La búsqueda MLP necesita al menos una configuración por modalidad.")
    seed_everything(fallback_config.seed)
    splits, routes = load_prepared_bundle(prepared_dir, prefix)
    rssi = detect_rssi_columns(splits["train"])
    pre = RSSIPreprocessor(use_mask=True).fit(splits["train"], rssi)
    x = {name: pre.transform(frame) for name, frame in splits.items()}
    y = {name: frame[TARGET_COLUMNS].to_numpy(dtype=float) for name, frame in splits.items()}
    target_scaler = StandardScaler().fit(y["train"])
    ys = {name: target_scaler.transform(values).astype(np.float32) for name, values in y.items()}
    clients = splits["train"]["CLIENT_ID"].astype(str).to_numpy()

    tuning_rows: List[Dict[str, object]] = []
    central_fitted: Dict[str, object] = {}
    federated_fitted: Dict[str, object] = {}
    central_by_id: Dict[str, MLPConfig] = {}
    federated_by_id: Dict[str, MLPConfig] = {}

    for index, candidate in enumerate(central_candidates):
        candidate_id = f"central_{index:02d}"
        seed_everything(candidate.seed)
        fitted = _fit_mlp_central(
            x["train"], ys["train"], x["val"], ys["val"], target_scaler, candidate
        )
        central_fitted[candidate_id] = fitted
        central_by_id[candidate_id] = candidate
        tuning_rows.append(
            {
                "model": "MLP",
                "task": "coordinates",
                "selection_modality": "centralized_global",
                "candidate_id": candidate_id,
                "selection_split": "val",
                "selection_metric": "rmse_2d_m",
                **coordinate_metrics(y["val"], fitted.predict(x["val"])),
                **asdict(candidate),
            }
        )

    for index, candidate in enumerate(federated_candidates):
        candidate_id = f"federated_{index:02d}"
        seed_everything(candidate.seed)
        fitted = _fit_mlp_fedavg(
            x["train"], ys["train"], clients, x["val"], ys["val"],
            target_scaler, candidate
        )
        federated_fitted[candidate_id] = fitted
        federated_by_id[candidate_id] = candidate
        tuning_rows.append(
            {
                "model": "MLP",
                "task": "coordinates",
                "selection_modality": "federated_global_fedavg",
                "candidate_id": candidate_id,
                "selection_split": "val",
                "selection_metric": "rmse_2d_m",
                **coordinate_metrics(y["val"], fitted.predict(x["val"])),
                **asdict(candidate),
            }
        )

    tuning_df = pd.DataFrame(tuning_rows)
    central_winner = (
        tuning_df[tuning_df["selection_modality"] == "centralized_global"]
        .sort_values(["rmse_2d_m", "candidate_id"])
        .iloc[0]
    )
    federated_winner = (
        tuning_df[tuning_df["selection_modality"] == "federated_global_fedavg"]
        .sort_values(["rmse_2d_m", "candidate_id"])
        .iloc[0]
    )
    central_id = str(central_winner["candidate_id"])
    federated_id = str(federated_winner["candidate_id"])
    tuning_df["selected"] = (
        ((tuning_df["selection_modality"] == "centralized_global")
         & (tuning_df["candidate_id"] == central_id))
        | ((tuning_df["selection_modality"] == "federated_global_fedavg")
           & (tuning_df["candidate_id"] == federated_id))
    )
    central_config = central_by_id[central_id]
    federated_config = federated_by_id[federated_id]
    central_global = central_fitted[central_id]
    fed_global = federated_fitted[federated_id]
    del central_fitted, federated_fitted
    central_extra = {
        "selected_candidate_id": central_id,
        "selected_from_n_candidates": len(central_candidates),
        "selection_metric": "val_rmse_2d_m",
        **asdict(central_config),
    }
    federated_extra = {
        "selected_candidate_id": federated_id,
        "selected_from_n_candidates": len(federated_candidates),
        "selection_metric": "val_rmse_2d_m",
        **asdict(federated_config),
    }
    rows: List[Dict[str, object]] = []
    for split in ["val", "test"]:
        rows.append(
            _metric_row(
                "MLP", "centralized_global", split, y[split], central_global.predict(x[split]),
                extra=central_extra,
            )
        )
        rows.append(
            _metric_row(
                "MLP", "federated_global_fedavg", split, y[split], fed_global.predict(x[split]),
                extra=federated_extra,
            )
        )

    available = sorted(pd.to_numeric(routes["N_CLUSTERS"]).astype(int).unique())
    cluster_values = list(cluster_values) if cluster_values is not None else available
    cluster_cache: Dict[int, Tuple[Dict[int, object], Dict[int, object]]] = {}
    cluster_val_rows: List[Dict[str, object]] = []
    for k_clusters in [int(k) for k in cluster_values if int(k) in available]:
        routed = {
            split: route_frame(splits[split], routes, split, k_clusters)
            for split in ["train", "val"]
        }
        c_models: Dict[int, object] = {}
        f_models: Dict[int, object] = {}
        for zone in sorted(routed["train"]["CLUSTER"].unique()):
            train_mask = routed["train"]["CLUSTER"].to_numpy() == zone
            val_mask = routed["val"]["CLUSTER"].to_numpy() == zone
            if np.sum(train_mask) < 8 or np.sum(val_mask) < 2:
                continue
            c_models[int(zone)] = _fit_mlp_central(
                x["train"][train_mask], ys["train"][train_mask],
                x["val"][val_mask], ys["val"][val_mask], target_scaler, central_config
            )
            f_models[int(zone)] = _fit_mlp_fedavg(
                x["train"][train_mask], ys["train"][train_mask], clients[train_mask],
                x["val"][val_mask], ys["val"][val_mask], target_scaler, federated_config
            )
        cluster_cache[k_clusters] = (c_models, f_models)
        labels = routed["val"]["CLUSTER"].to_numpy(dtype=int)
        cpred, ccov = _predict_by_zone(c_models, central_global, x["val"], labels)
        fpred, fcov = _predict_by_zone(f_models, fed_global, x["val"], labels)
        gate_acc = float(np.mean(labels == routed["val"]["CLUSTER_ORACLE"].to_numpy()))
        cluster_val_rows.extend(
            [
                _metric_row(
                    "MLP", "centralized_by_predicted_cluster", "val", y["val"], cpred,
                    n_clusters=k_clusters,
                    extra={"coverage": ccov, "gate_accuracy_diagnostic": gate_acc, **central_extra},
                ),
                _metric_row(
                    "MLP", "federated_by_predicted_cluster_fedavg", "val", y["val"], fpred,
                    n_clusters=k_clusters,
                    extra={"coverage": fcov, "gate_accuracy_diagnostic": gate_acc, **federated_extra},
                ),
            ]
        )
    cluster_tuning_df = pd.DataFrame(cluster_val_rows)
    for modality in [
        "centralized_by_predicted_cluster",
        "federated_by_predicted_cluster_fedavg",
    ]:
        val_rows = cluster_tuning_df[cluster_tuning_df["modality"] == modality]
        if not val_rows.empty:
            best_k = int(val_rows.sort_values("rmse_2d_m").iloc[0]["n_clusters"])
            rows.append(val_rows[val_rows["n_clusters"] == best_k].iloc[0].to_dict())
            test_routed = route_frame(splits["test"], routes, "test", best_k)
            labels = test_routed["CLUSTER"].to_numpy(dtype=int)
            c_models, f_models = cluster_cache[best_k]
            if modality == "centralized_by_predicted_cluster":
                pred, coverage = _predict_by_zone(c_models, central_global, x["test"], labels)
            else:
                pred, coverage = _predict_by_zone(f_models, fed_global, x["test"], labels)
            rows.append(
                _metric_row(
                    "MLP", modality, "test", y["test"], pred, n_clusters=best_k,
                    extra={
                        "coverage": coverage,
                        "gate_accuracy_diagnostic": float(
                            np.mean(labels == test_routed["CLUSTER_ORACLE"].to_numpy())
                        ),
                        **(
                            central_extra
                            if modality == "centralized_by_predicted_cluster"
                            else federated_extra
                        ),
                    },
                )
            )
    keep = pd.DataFrame(rows).sort_values(["split", "modality"]).reset_index(drop=True)
    if output_csv is not None:
        Path(output_csv).parent.mkdir(parents=True, exist_ok=True)
        keep.to_csv(output_csv, index=False)
        tuning_df.to_csv(
            Path(output_csv).with_name(
                Path(output_csv).stem + "_hyperparameter_tuning.csv"
            ),
            index=False,
        )
        cluster_tuning_df.to_csv(
            Path(output_csv).with_name(Path(output_csv).stem + "_cluster_tuning.csv"), index=False
        )
        selection_path = Path(output_csv).with_name(
            Path(output_csv).stem + "_selected_hyperparameters.json"
        )
        with selection_path.open("w", encoding="utf-8") as handle:
            json.dump(
                {
                    "model": "MLP",
                    "task": "coordinates",
                    "selection_split": "validation",
                    "selection_metric": "rmse_2d_m",
                    "centralized_global": {
                        "candidate_id": central_id,
                        **asdict(central_config),
                    },
                    "federated_global_fedavg": {
                        "candidate_id": federated_id,
                        **asdict(federated_config),
                    },
                    "cluster_protocol": (
                        "The centralized and federated winners are reused in their "
                        "respective cluster-aware strategies; K is selected on validation."
                    ),
                },
                handle,
                indent=2,
                ensure_ascii=False,
            )
    return keep


## 10. Espacio de hiperparámetros


In [15]:
COMMON = dict(
    weight_decay=1e-5,
    batch_size=128,
    central_epochs=100,
    patience=15,
    federated_rounds=40,
    local_epochs=2,
    force_cpu=False,
    seed=42,
)

CENTRAL_CONFIGS = [
    MLPConfig(
        hidden_sizes=hidden,
        dropout=dropout,
        learning_rate=learning_rate,
        client_weighting="uniform",
        **COMMON,
    )
    for hidden, dropout, learning_rate in product(
        ((128, 64), (256, 128)),
        (0.0, 0.10),
        (1e-3, 5e-4),
    )
]

FEDERATED_CONFIGS = [
    MLPConfig(
        hidden_sizes=hidden,
        dropout=0.10,
        learning_rate=learning_rate,
        client_weighting=client_weighting,
        **COMMON,
    )
    for hidden, learning_rate, client_weighting in product(
        ((128, 64), (256, 128)),
        (1e-3, 5e-4),
        ("uniform", "samples"),
    )
]

TUNING_DISPLAY_COLUMNS = [
    "selection_modality",
    "candidate_id",
    "selected",
    "rmse_2d_m",
    "mean_error_m",
    "hidden_sizes",
    "dropout",
    "learning_rate",
    "batch_size",
    "central_epochs",
    "federated_rounds",
    "local_epochs",
    "client_weighting",
]
FLOOR_TUNING_DISPLAY_COLUMNS = [
    "selection_modality",
    "candidate_id",
    "selected",
    "floor_accuracy",
    "hidden_sizes",
    "dropout",
    "learning_rate",
    "batch_size",
    "central_epochs",
    "federated_rounds",
    "local_epochs",
    "client_weighting",
]

print(
    "Candidatos MLP:",
    len(CENTRAL_CONFIGS),
    "centrales y",
    len(FEDERATED_CONFIGS),
    "federados",
)


Candidatos MLP: 8 centrales y 8 federados


In [16]:
central_grid = pd.DataFrame(
    [{"candidate_id": f"central_{index:02d}", **asdict(config)} for index, config in enumerate(CENTRAL_CONFIGS)]
)
federated_grid = pd.DataFrame(
    [{"candidate_id": f"federated_{index:02d}", **asdict(config)} for index, config in enumerate(FEDERATED_CONFIGS)]
)

print("=== Rejilla centralizada ===")
display(central_grid)
print("=== Rejilla federada ===")
display(federated_grid)
print("Valores de K espacial evaluados:", CLUSTER_VALUES)


=== Rejilla centralizada ===


,candidate_id,hidden_sizes,dropout,learning_rate,weight_decay,batch_size,central_epochs,patience,federated_rounds,local_epochs,client_weighting,force_cpu,seed
0,central_00,"(128, 64)",0.0,0.0010,0.00001,128,100,15,40,2,uniform,False,42
1,central_01,"(128, 64)",0.0,0.0005,0.00001,128,100,15,40,2,uniform,False,42
2,central_02,"(128, 64)",0.1,0.0010,0.00001,128,100,15,40,2,uniform,False,42
3,central_03,"(128, 64)",0.1,0.0005,0.00001,128,100,15,40,2,uniform,False,42
4,central_04,"(256, 128)",0.0,0.0010,0.00001,128,100,15,40,2,uniform,False,42
5,central_05,"(256, 128)",0.0,0.0005,0.00001,128,100,15,40,2,uniform,False,42
6,central_06,"(256, 128)",0.1,0.0010,0.00001,128,100,15,40,2,uniform,False,42
7,central_07,"(256, 128)",0.1,0.0005,0.00001,128,100,15,40,2,uniform,False,42


=== Rejilla federada ===


,candidate_id,hidden_sizes,dropout,learning_rate,weight_decay,batch_size,central_epochs,patience,federated_rounds,local_epochs,client_weighting,force_cpu,seed
0,federated_00,"(128, 64)",0.1,0.0010,0.00001,128,100,15,40,2,uniform,False,42
1,federated_01,"(128, 64)",0.1,0.0010,0.00001,128,100,15,40,2,samples,False,42
2,federated_02,"(128, 64)",0.1,0.0005,0.00001,128,100,15,40,2,uniform,False,42
3,federated_03,"(128, 64)",0.1,0.0005,0.00001,128,100,15,40,2,samples,False,42
4,federated_04,"(256, 128)",0.1,0.0010,0.00001,128,100,15,40,2,uniform,False,42
5,federated_05,"(256, 128)",0.1,0.0010,0.00001,128,100,15,40,2,samples,False,42
6,federated_06,"(256, 128)",0.1,0.0005,0.00001,128,100,15,40,2,uniform,False,42
7,federated_07,"(256, 128)",0.1,0.0005,0.00001,128,100,15,40,2,samples,False,42


Valores de K espacial evaluados: [2, 3, 4, 5, 6, 7, 8, 9, 10]


## 11. Estrategias de entrenamiento


### 11.1. Modelos globales

La modalidad centralizada ajusta una única red con todas las muestras de
train. La modalidad federada mantiene las muestras separadas por
`CLIENT_ID` y combina parámetros mediante FedAvg. Cada modalidad elige su
propia configuración usando únicamente validación.


### 11.2. Modelos por clúster

El ganador centralizado y el ganador federado se reutilizan dentro de sus
respectivas estrategias por clúster. El número de clústeres se selecciona
por modalidad con validación. El router asigna validación y test a partir
del RSSI; `CLUSTER_ORACLE` se conserva solo como diagnóstico.


## 12. Entrenamiento y evaluación de coordenadas


In [17]:
coordinate_results = run_mlp_experiment(
    prepared_dir=PREPARED_DIR,
    prefix=PREFIX,
    central_configs=CENTRAL_CONFIGS,
    federated_configs=FEDERATED_CONFIGS,
    cluster_values=CLUSTER_VALUES,
    output_csv=COORD_RESULTS_PATH,
)
print("Experimento de coordenadas finalizado.")


Experimento de coordenadas finalizado.


### 12.1. Búsqueda global con validación


In [18]:
coordinate_tuning = pd.read_csv(COORD_HPARAM_TUNING_PATH)
coordinate_tuning = coordinate_tuning.sort_values(
    ["selection_modality", "rmse_2d_m", "candidate_id"]
).reset_index(drop=True)

coordinate_tuning_columns = [
    column
    for column in TUNING_DISPLAY_COLUMNS
    if column in coordinate_tuning.columns
]
print("La selección global se realiza con RMSE radial 2D de validación.")
display(coordinate_tuning[coordinate_tuning_columns])


La selección global se realiza con RMSE radial 2D de validación.


,selection_modality,candidate_id,selected,rmse_2d_m,mean_error_m,hidden_sizes,dropout,learning_rate,batch_size,central_epochs,federated_rounds,local_epochs,client_weighting
0,centralized_global,central_04,True,2.596693,2.170402,"(256, 128)",0.0,0.0010,128,100,40,2,uniform
1,centralized_global,central_06,False,2.716829,2.239536,"(256, 128)",0.1,0.0010,128,100,40,2,uniform
2,centralized_global,central_05,False,2.780359,2.311615,"(256, 128)",0.0,0.0005,128,100,40,2,uniform
3,centralized_global,central_00,False,2.823525,2.289940,"(128, 64)",0.0,0.0010,128,100,40,2,uniform
4,centralized_global,central_01,False,2.963271,2.375117,"(128, 64)",0.0,0.0005,128,100,40,2,uniform
5,centralized_global,central_07,False,3.062104,2.477656,"(256, 128)",0.1,0.0005,128,100,40,2,uniform
6,centralized_global,central_02,False,3.104966,2.537681,"(128, 64)",0.1,0.0010,128,100,40,2,uniform
7,centralized_global,central_03,False,3.956311,3.095406,"(128, 64)",0.1,0.0005,128,100,40,2,uniform
8,federated_global_fedavg,federated_07,True,17.488527,15.300041,"(256, 128)",0.1,0.0005,128,100,40,2,samples
9,federated_global_fedavg,federated_06,False,18.235687,15.957706,"(256, 128)",0.1,0.0005,128,100,40,2,uniform


### 12.2. Selección del número de clústeres


In [19]:
coordinate_cluster_tuning = pd.read_csv(COORD_CLUSTER_TUNING_PATH)
coordinate_cluster_tuning = coordinate_cluster_tuning.sort_values(
    ["modality", "rmse_2d_m", "n_clusters"]
).reset_index(drop=True)

display(
    coordinate_cluster_tuning[
        [
            "modality",
            "n_clusters",
            "rmse_2d_m",
            "mean_error_m",
            "coverage",
            "gate_accuracy_diagnostic",
            "selected_candidate_id",
        ]
    ]
)


,modality,n_clusters,rmse_2d_m,mean_error_m,coverage,gate_accuracy_diagnostic,selected_candidate_id
0,centralized_by_predicted_cluster,9,2.866884,2.468225,1.0,0.952047,central_04
1,centralized_by_predicted_cluster,4,3.028059,2.381169,1.0,0.971345,central_04
2,centralized_by_predicted_cluster,2,3.071241,2.459220,1.0,0.984211,central_04
3,centralized_by_predicted_cluster,6,3.122170,2.523682,1.0,0.939766,central_04
4,centralized_by_predicted_cluster,7,3.279146,2.651754,1.0,0.949123,central_04
5,centralized_by_predicted_cluster,8,3.356504,2.751394,1.0,0.933333,central_04
6,centralized_by_predicted_cluster,10,3.515727,2.749710,1.0,0.943275,central_04
7,centralized_by_predicted_cluster,5,3.543530,2.750873,1.0,0.955556,central_04
8,centralized_by_predicted_cluster,3,3.839933,2.786992,1.0,0.984211,central_04
9,federated_by_predicted_cluster_fedavg,10,3.235538,2.601933,1.0,0.943275,federated_07


### 12.3. Resultados de validación y test


In [20]:
print("=== Coordenadas: validación ===")
display(compact_results(coordinate_results, split="val"))

print("=== Coordenadas: test final ===")
display(compact_results(coordinate_results, split="test"))


=== Coordenadas: validación ===


,model,modality,n_clusters,rmse_2d_m,mean_error_m,median_error_m,p75_error_m,p95_error_m,coverage,gate_accuracy_diagnostic
0,MLP,centralized_global,NaN,2.596693,2.170402,1.809459,2.794155,5.221804,NaN,NaN
1,MLP,centralized_by_predicted_cluster,9.0,2.866884,2.468225,2.195137,3.340032,5.074484,1.0,0.952047
2,MLP,federated_by_predicted_cluster_fedavg,10.0,3.235538,2.601933,2.118498,3.243620,7.419313,1.0,0.943275
3,MLP,federated_global_fedavg,NaN,17.488527,15.300041,12.398732,16.190305,37.418759,NaN,NaN


=== Coordenadas: test final ===


,model,modality,n_clusters,rmse_2d_m,mean_error_m,median_error_m,p75_error_m,p95_error_m,coverage,gate_accuracy_diagnostic
0,MLP,centralized_global,NaN,3.114812,2.505839,2.239163,3.266552,5.430857,NaN,NaN
1,MLP,centralized_by_predicted_cluster,9.0,4.018593,3.175243,2.549796,4.115484,7.342380,1.0,0.945349
2,MLP,federated_by_predicted_cluster_fedavg,10.0,4.740847,3.641704,2.815410,4.984448,8.690420,1.0,0.920930
3,MLP,federated_global_fedavg,NaN,16.994522,14.828748,11.947260,15.483160,39.001019,NaN,NaN


### 12.4. Hiperparámetros seleccionados


In [21]:
with COORD_SELECTED_PATH.open("r", encoding="utf-8") as handle:
    selected_coordinate_hyperparameters = json.load(handle)

print(json.dumps(selected_coordinate_hyperparameters, indent=2, ensure_ascii=False))


{
  "model": "MLP",
  "task": "coordinates",
  "selection_split": "validation",
  "selection_metric": "rmse_2d_m",
  "centralized_global": {
    "candidate_id": "central_04",
    "hidden_sizes": [
      256,
      128
    ],
    "dropout": 0.0,
    "learning_rate": 0.001,
    "weight_decay": 1e-05,
    "batch_size": 128,
    "central_epochs": 100,
    "patience": 15,
    "federated_rounds": 40,
    "local_epochs": 2,
    "client_weighting": "uniform",
    "force_cpu": false,
    "seed": 42
  },
  "federated_global_fedavg": {
    "candidate_id": "federated_07",
    "hidden_sizes": [
      256,
      128
    ],
    "dropout": 0.1,
    "learning_rate": 0.0005,
    "weight_decay": 1e-05,
    "batch_size": 128,
    "central_epochs": 100,
    "patience": 15,
    "federated_rounds": 40,
    "local_epochs": 2,
    "client_weighting": "samples",
    "force_cpu": false,
    "seed": 42
  },
  "cluster_protocol": "The centralized and federated winners are reused in their respective cluster-aware s

## 13. Resumen final


In [22]:
print("=== Resumen final de coordenadas ===")
display(compact_results(coordinate_results, split="test"))

if FLOOR_TASK:
    print("=== Resumen final de planta ===")
    display(compact_floor_results(floor_results, split="test"))


=== Resumen final de coordenadas ===


,model,modality,n_clusters,rmse_2d_m,mean_error_m,median_error_m,p75_error_m,p95_error_m,coverage,gate_accuracy_diagnostic
0,MLP,centralized_global,NaN,3.114812,2.505839,2.239163,3.266552,5.430857,NaN,NaN
1,MLP,centralized_by_predicted_cluster,9.0,4.018593,3.175243,2.549796,4.115484,7.342380,1.0,0.945349
2,MLP,federated_by_predicted_cluster_fedavg,10.0,4.740847,3.641704,2.815410,4.984448,8.690420,1.0,0.920930
3,MLP,federated_global_fedavg,NaN,16.994522,14.828748,11.947260,15.483160,39.001019,NaN,NaN


## 14. Ficheros generados


In [23]:
generated_files = [
    COORD_RESULTS_PATH,
    COORD_HPARAM_TUNING_PATH,
    COORD_CLUSTER_TUNING_PATH,
    COORD_SELECTED_PATH,
]
if FLOOR_TASK:
    generated_files.extend(
        [
            FLOOR_RESULTS_PATH,
            FLOOR_HPARAM_TUNING_PATH,
            FLOOR_CLUSTER_TUNING_PATH,
            FLOOR_SELECTED_PATH,
        ]
    )

display(
    pd.DataFrame(
        [
            {
                "file": str(path.relative_to(ROOT)),
                "exists": path.exists(),
                "size_bytes": path.stat().st_size if path.exists() else 0,
            }
            for path in generated_files
        ]
    )
)


,file,exists,size_bytes
0,results/SOD/sod_raw_mlp_corrected.csv,True,2258
1,results/SOD/sod_raw_mlp_corrected_hyperparamet...,True,3955
2,results/SOD/sod_raw_mlp_corrected_cluster_tuni...,True,4955
3,results/SOD/sod_raw_mlp_corrected_selected_hyp...,True,1042
